> Repository edition: run from the repository root using an environment with the dependencies in `requirements.txt`. Machine-specific environment paths have been replaced. Historical outputs are retained except environment checks; they are not evidence of a fresh rerun. See `docs/REPRODUCIBILITY.md`.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

# Define the data locations for round 6.
initial_data_path = Path("initial_data")
feedback_path = Path("week_5")

# Check the Python environment and required files.
print("Python executable:", sys.executable)
print("Working directory:", Path.cwd())

print("Initial data folder found:", initial_data_path.is_dir())
print(
    "Cumulative inputs found:",
    (feedback_path / "inputs.txt").is_file()
)
print(
    "Cumulative outputs found:",
    (feedback_path / "outputs.txt").is_file()
)

assert initial_data_path.is_dir(), "Missing initial_data folder."
assert (feedback_path / "inputs.txt").is_file(), (
    "Missing week_5/inputs.txt."
)
assert (feedback_path / "outputs.txt").is_file(), (
    "Missing week_5/outputs.txt."
)

Python executable: c:\Users\AG\BeatrizGIT\Master\.venv\Scripts\python.exe
Working directory: c:\Users\AG\BeatrizGIT\Master\CapstoneProject
Initial data folder found: True
Cumulative inputs found: True
Cumulative outputs found: True


### Environment and data-path checks

This cell displays the active Python interpreter and working directory.
It checks that the initial data folder and the cumulative evaluation
results from week 5 are available.

The new files contain five completed rounds. These observations will
inform the sixth-round queries for Module 17.

The assertions stop execution if a required folder or file is missing.
Successful checks confirm file availability; the next step will
validate the contents and rebuild each function's cumulative data set.

> *Methodological note: The weekly files contain cumulative results.
> We will combine the initial observations with the week 5 snapshot
> once, rather than append every weekly file and duplicate observations.*

In [2]:
import ast
import re


# Read cumulative results without executing file contents.
def read_feedback_rounds(path):
    text = path.read_text(encoding="utf-8-sig").strip()

    # Convert NumPy-style array([...]) representations to lists.
    text = re.sub(
        r"array\((\[[^\[\]]*\])\)",
        r"\1",
        text
    )

    tree = ast.parse(text, mode="exec")
    rounds = []

    for node in tree.body:
        if not isinstance(node, ast.Expr):
            raise ValueError(f"Unexpected content in {path}")

        rounds.append(ast.literal_eval(node.value))

    return rounds


# Load the five completed rounds.
input_rounds = read_feedback_rounds(
    feedback_path / "inputs.txt"
)
output_rounds = read_feedback_rounds(
    feedback_path / "outputs.txt"
)

completed_rounds = 5

assert len(input_rounds) == len(output_rounds) == completed_rounds, (
    "Expected five cumulative rounds in each file."
)

for round_inputs, round_outputs in zip(input_rounds, output_rounds):
    assert len(round_inputs) == len(round_outputs) == 8, (
        "Each round must contain eight input-output pairs."
    )

expected_dimensions = {
    1: 2, 2: 2, 3: 3, 4: 4,
    5: 4, 6: 5, 7: 6, 8: 8
}

initial_data = {}
updated_data = {}

for function in range(1, 9):
    folder = initial_data_path / f"function_{function}"

    X_initial = np.load(
        folder / "initial_inputs.npy",
        allow_pickle=False
    )
    y_initial = np.load(
        folder / "initial_outputs.npy",
        allow_pickle=False
    ).reshape(-1)

    dimensions = expected_dimensions[function]

    assert X_initial.ndim == 2
    assert X_initial.shape == (len(y_initial), dimensions)
    assert np.isfinite(X_initial).all()
    assert np.isfinite(y_initial).all()

    X_feedback = np.asarray(
        [row[function - 1] for row in input_rounds],
        dtype=float
    )
    y_feedback = np.asarray(
        [row[function - 1] for row in output_rounds],
        dtype=float
    )

    assert X_feedback.shape == (completed_rounds, dimensions)
    assert y_feedback.shape == (completed_rounds,)
    assert np.isfinite(X_feedback).all()
    assert np.isfinite(y_feedback).all()
    assert ((X_feedback >= 0) & (X_feedback < 1)).all()

    initial_data[function] = {
        "X": X_initial.copy(),
        "y": y_initial.copy()
    }

    # Rebuild from source files each time to avoid repeated additions.
    updated_data[function] = {
        "X": np.vstack([X_initial, X_feedback]),
        "y": np.concatenate([y_initial, y_feedback])
    }

    print(
        f"Function {function}: "
        f"{len(y_initial)} → "
        f"{len(updated_data[function]['y'])} observations "
        f"| {dimensions} variables"
    )

Function 1: 10 → 15 observations | 2 variables
Function 2: 10 → 15 observations | 2 variables
Function 3: 15 → 20 observations | 3 variables
Function 4: 30 → 35 observations | 4 variables
Function 5: 20 → 25 observations | 4 variables
Function 6: 20 → 25 observations | 5 variables
Function 7: 30 → 35 observations | 6 variables
Function 8: 40 → 45 observations | 8 variables


### Loading and validating five rounds of evaluation results

This cell loads each function's initial observations and the five
completed rounds contained in the cumulative week 5 files.

It checks that every round contains eight input-output pairs and
validates array dimensions, finite values and submitted coordinate
bounds. The files are parsed as literal data without executing their
contents.

Each data set is rebuilt from the original observations and the
selected cumulative snapshot. Rerunning this cell therefore does not
append the same evaluations repeatedly.

The expected observation counts for Functions 1–8 are:
15, 15, 20, 35, 25, 25, 35 and 45. Only Functions 1 and 2 started with
ten observations; the larger initial data sets are retained in full.

These data will support the sixth-round queries. The initial
observations are also stored separately so that subsequent analysis
can compare progress against the original best outputs.

> *Methodological note: Each row of the input array must remain paired
> with its corresponding output. Both successful and unsuccessful
> evaluations are retained, since both provide information about the
> unknown function.*

In [3]:
# Track progress across all five completed rounds.
progress_rows = []

for function in range(1, 9):
    initial_best = float(np.max(initial_data[function]["y"]))
    best_so_far = initial_best

    row = {
        "Function": function,
        "Initial best": initial_best
    }

    for round_number, round_outputs in enumerate(
        output_rounds, start=1
    ):
        observed_output = float(round_outputs[function - 1])

        improved = observed_output > best_so_far
        improvement = max(0.0, observed_output - best_so_far)

        row[f"Round {round_number} output"] = observed_output
        row[f"Round {round_number} improved"] = improved
        row[f"Round {round_number} improvement"] = improvement

        best_so_far = max(best_so_far, observed_output)

    row["Current best"] = best_so_far
    progress_rows.append(row)

progress_summary = (
    pd.DataFrame(progress_rows)
    .set_index("Function")
)

# Compare the latest evaluation with the best preceding observation.
round_5_summary = pd.DataFrame(
    {
        "Best before round 5": [
            float(np.max(updated_data[f]["y"][:-1]))
            for f in range(1, 9)
        ],
        "Round 5 output": progress_summary["Round 5 output"],
        "Round 5 improved": progress_summary["Round 5 improved"],
        "Improvement": progress_summary["Round 5 improvement"],
        "Current best": progress_summary["Current best"]
    },
    index=progress_summary.index
)

display(
    round_5_summary.style.format({
        "Best before round 5": "{:.6e}",
        "Round 5 output": "{:.6e}",
        "Improvement": "{:.6e}",
        "Current best": "{:.6e}"
    })
)

for round_number in range(1, completed_rounds + 1):
    improved_functions = progress_summary.index[
        progress_summary[f"Round {round_number} improved"]
    ].tolist()

    print(
        f"Functions improved in round {round_number}: "
        f"{improved_functions} | Count: {len(improved_functions)}"
    )

,Best before round 5,Round 5 output,Round 5 improved,Improvement,Current best
Function,,,,,
1,7.710875e-16,2.547631e-31,False,0.000000e+00,7.710875e-16
2,6.582395e-01,-8.152648e-02,False,0.000000e+00,6.582395e-01
3,-3.483531e-02,-3.096391e-02,True,3.871400e-03,-3.096391e-02
4,1.948721e-01,4.098217e-01,True,2.149496e-01,4.098217e-01
5,4.215408e+03,4.779885e+03,True,5.644768e+02,4.779885e+03
6,-3.843809e-01,-9.485219e-01,False,0.000000e+00,-3.843809e-01
7,2.097864e+00,1.580747e+00,False,0.000000e+00,2.097864e+00
8,9.915965e+00,9.922955e+00,True,6.989080e-03,9.922955e+00


Functions improved in round 1: [2, 4, 5, 7, 8] | Count: 5
Functions improved in round 2: [6, 8] | Count: 2
Functions improved in round 3: [5, 8] | Count: 2
Functions improved in round 4: [6, 8] | Count: 2
Functions improved in round 5: [3, 4, 5, 8] | Count: 4


### Tracking optimisation progress through round 5

This cell compares each evaluation with the best output available
before that round. All eight functions are maximisation problems,
so a larger output is better, including when values are negative.

The full history is stored in `progress_summary`. The displayed
`round_5_summary` focuses on the latest evaluation for each function.

The improvement column measures the increase in the best observed
output. It is zero when the new evaluation does not establish a new
best value; it does not represent the signed change in output.

Round 5 improved Functions 3, 4, 5 and 8. Function 3 obtained a less
negative output, while Functions 4, 5 and 8 exceeded their previous
positive maxima. The remaining evaluations are retained because
they also provide information about the response surfaces.

Four functions improved in this round, compared with two in round 4.
However, this alone does not establish that the revised strategy is
superior: the functions, query locations and accumulated information
differ, and only one new evaluation is available per function.

> *Interpretation note: Improvement should be assessed within each
> function because output scales differ. A small improvement in
> Function 8 cannot be compared directly with a numerically larger
> improvement in Function 5.*

In [4]:
# Historical GP predictions from CP_16, recorded before round 5 results.
# Values are retained at the precision displayed in that notebook.
round_5_predictions = {
    1: {"mean": -2.299355e-04, "std": 9.865848e-04},
    2: {"mean": 2.517883e-01, "std": 2.357109e-01},
    3: {"mean": -5.621860e-02, "std": 6.718938e-02},
    4: {"mean": -8.363377e-02, "std": 2.990839e-01},
    5: {"mean": 4.282913e+03, "std": 2.279114e+02},
    6: {"mean": -5.450000e-01, "std": 1.717249e-01},
    7: {"mean": 2.030465e+00, "std": 1.377804e-01},
    8: {"mean": 1.000922e+01, "std": 6.393926e-02}
}

round_5_expected_inputs = {
    1: [0.795811, 0.503445],
    2: [0.000148, 0.977716],
    3: [0.999318, 0.964676, 0.505507],
    4: [0.410712, 0.405265, 0.384089, 0.433507],
    5: [0.596309, 0.996343, 0.995719, 0.998368],
    6: [0.548648, 0.528095, 0.973039, 0.990992, 0.036642],
    7: [0.006389, 0.227533, 0.638856, 0.045721, 0.344255, 0.779122],
    8: [
        0.167821, 0.083323, 0.143724, 0.015688,
        0.998566, 0.507939, 0.294815, 0.865301
    ]
}

# Auxiliary predictions recorded for the selected points.
round_5_auxiliary_predictions = {
    5: 4.411068e+03,  # One-hidden-layer neural network
    7: 7.397378e-01   # Linear regression
}

prediction_rows = []

for function in range(1, 9):
    actual_input = np.asarray(
        input_rounds[4][function - 1],
        dtype=float
    )

    # Ensure predictions are compared with the correct evaluated point.
    assert np.allclose(
        actual_input,
        round_5_expected_inputs[function],
        rtol=0,
        atol=1e-9
    ), f"Round 5 input mismatch for Function {function}"

    observed = float(output_rounds[4][function - 1])
    mean = round_5_predictions[function]["mean"]
    std = round_5_predictions[function]["std"]

    assert std > 0

    auxiliary = round_5_auxiliary_predictions.get(function, np.nan)

    prediction_rows.append({
        "Function": function,
        "Predicted mean": mean,
        "Predicted std": std,
        "Observed output": observed,
        "Observed minus predicted": observed - mean,
        "Absolute error / predicted std": abs(observed - mean) / std,
        "GP absolute error": abs(observed - mean),
        "Auxiliary prediction": auxiliary,
        "Auxiliary absolute error": abs(observed - auxiliary)
    })

prediction_review = (
    pd.DataFrame(prediction_rows)
    .set_index("Function")
)

display(
    prediction_review.style.format({
        "Predicted mean": "{:.6e}",
        "Predicted std": "{:.6e}",
        "Observed output": "{:.6e}",
        "Observed minus predicted": "{:.6e}",
        "Absolute error / predicted std": "{:.2f}",
        "GP absolute error": "{:.6e}",
        "Auxiliary prediction": "{:.6e}",
        "Auxiliary absolute error": "{:.6e}"
    }, na_rep="—")
)

,Predicted mean,Predicted std,Observed output,Observed minus predicted,Absolute error / predicted std,GP absolute error,Auxiliary prediction,Auxiliary absolute error
Function,,,,,,,,
1,-2.299355e-04,9.865848e-04,2.547631e-31,2.299355e-04,0.23,2.299355e-04,—,—
2,2.517883e-01,2.357109e-01,-8.152648e-02,-3.333148e-01,1.41,3.333148e-01,—,—
3,-5.621860e-02,6.718938e-02,-3.096391e-02,2.525469e-02,0.38,2.525469e-02,—,—
4,-8.363377e-02,2.990839e-01,4.098217e-01,4.934555e-01,1.65,4.934555e-01,—,—
5,4.282913e+03,2.279114e+02,4.779885e+03,4.969721e+02,2.18,4.969721e+02,4.411068e+03,3.688171e+02
6,-5.450000e-01,1.717249e-01,-9.485219e-01,-4.035219e-01,2.35,4.035219e-01,—,—
7,2.030465e+00,1.377804e-01,1.580747e+00,-4.497183e-01,3.26,4.497183e-01,7.397378e-01,8.410089e-01
8,1.000922e+01,6.393926e-02,9.922955e+00,-8.626544e-02,1.35,8.626544e-02,—,—


### Reviewing predictions against round 5 outcomes

This cell compares the predictions recorded in CP_16 with the actual
outputs returned for the fifth-round queries. It verifies the input
coordinates before matching each prediction to its observed result.

The signed error indicates whether the GP underestimated or
overestimated the output. The absolute error divided by the predicted
standard deviation measures the discrepancy relative to the model's
reported uncertainty.

For Functions 5 and 7, the table also compares the historical
auxiliary predictions with the observed outputs. This allows us to
assess whether the neural network and linear regression provided
useful additional information at the selected points.

These predictions were made before the fifth-round outputs became
available. They are not reconstructed by fitting models to the newly
updated data.

Optimisation progress and prediction accuracy are assessed separately.
A query can improve the best observed output even when the model
overpredicts it, or produce an accurately predicted result without
improving the current maximum.

> *Methodological note: One new evaluation per function cannot establish
> uncertainty calibration or overall model superiority. Auxiliary
> errors should be compared with GP errors for the same function and
> evaluated point. The recorded predictions have limited display
> precision, so the calculated errors are approximate.*

### Interpretation of round 5 prediction errors

The fifth-round results improved Functions 3, 4, 5 and 8, but the
prediction review reveals different levels of reliability.

For Function 5, the auxiliary neural network was closer to the observed
output than the GP: its absolute error was approximately 369 compared
with 497. Both models underestimated the improvement. This supports
retaining the network as a supplementary diagnostic, without concluding
that it is generally superior.

For Function 7, the GP was closer than linear regression, despite
overestimating the output by 3.26 predicted standard deviations.
The earlier cross-validation advantage of linear regression therefore
did not translate into a more accurate prediction at this particular
query. Average predictive performance and accuracy near promising
regions are distinct considerations.

Function 6's exploratory query did not improve the current best, and
its error was 2.35 predicted standard deviations. This motivates
comparing further global proposals with candidates near its best
observed input, rather than assuming that uncertainty bonuses identify
reliable opportunities.

Function 8 improved its maximum despite falling below the predicted
mean. Function 1 again produced an output close to zero, providing
little evidence that the model has identified a useful maximum.

For the next round, I will initially retain the previous model
configurations and refit them with the new observations. This keeps
the update interpretable before considering further changes.

> *Module 17 connection: CNNs exploit meaningful local structure and
> shared patterns across positions. Our input coordinates have no
> established spatial or sequential structure of this kind. Reshaping
> them into an image would not establish such structure, so a CNN is
> not justified solely because this module introduces convolutions.*

In [5]:
from time import perf_counter

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel, Matern, WhiteKernel
)
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.compose import TransformedTargetRegressor


# Retain the reference configurations used in CP_16.
gp_upper_bounds = {
    1: 2.0, 2: 2.0, 3: 2.0, 4: 2.0,
    5: 2.0, 6: 2.0, 7: 2.0, 8: 10.0
}

round_6_gp_models = {}
round_6_auxiliary_models = {}
fit_rows = []

for function in range(1, 9):
    X = updated_data[function]["X"]
    y = updated_data[function]["y"]

    upper_bound = gp_upper_bounds[function]

    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * X.shape[1],
            length_scale_bounds=(0.02, upper_bound),
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    start = perf_counter()
    gp.fit(X, y)
    gp_seconds = perf_counter() - start

    round_6_gp_models[function] = gp

    auxiliary_name = "None"
    auxiliary_seconds = np.nan

    if function in (5, 7):
        if function == 5:
            estimator = MLPRegressor(
                hidden_layer_sizes=(8,),
                activation="tanh",
                solver="lbfgs",
                alpha=0.1,
                max_iter=2000,
                random_state=42
            )
            auxiliary_name = "NN (8,)"
        else:
            estimator = LinearRegression()
            auxiliary_name = "Linear regression"

        auxiliary = TransformedTargetRegressor(
            regressor=make_pipeline(
                StandardScaler(),
                estimator
            ),
            transformer=StandardScaler()
        )

        start = perf_counter()
        auxiliary.fit(X, y)
        auxiliary_seconds = perf_counter() - start

        round_6_auxiliary_models[function] = auxiliary

    fit_rows.append({
        "Function": function,
        "Observations": len(y),
        "Length-scale upper bound": upper_bound,
        "Fitted kernel": str(gp.kernel_),
        "GP fit seconds": gp_seconds,
        "Auxiliary model": auxiliary_name,
        "Auxiliary fit seconds": auxiliary_seconds
    })

round_6_fit_summary = (
    pd.DataFrame(fit_rows)
    .set_index("Function")
)

display(
    round_6_fit_summary.style.format({
        "Length-scale upper bound": "{:.1f}",
        "GP fit seconds": "{:.3f}",
        "Auxiliary fit seconds": "{:.3f}"
    }, na_rep="—")
)

c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 0.02. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-pac

,Observations,Length-scale upper bound,Fitted kernel,GP fit seconds,Auxiliary model,Auxiliary fit seconds
Function,,,,,,
1,15,2.0,"1.07**2 * Matern(length_scale=[0.02, 2], nu=2.5) + WhiteKernel(noise_level=1.12e-09)",0.671,None,—
2,15,2.0,"0.932**2 * Matern(length_scale=[0.0698, 2], nu=2.5) + WhiteKernel(noise_level=0.00531)",0.361,None,—
3,20,2.0,"1.15**2 * Matern(length_scale=[2, 1.63, 0.0641], nu=2.5) + WhiteKernel(noise_level=1.27e-06)",0.506,None,—
4,35,2.0,"2.85**2 * Matern(length_scale=[1.62, 1.54, 1.52, 1.51], nu=2.5) + WhiteKernel(noise_level=0.000929)",0.770,None,—
5,25,2.0,"1.77**2 * Matern(length_scale=[1.05, 1.31, 0.817, 1.37], nu=2.5) + WhiteKernel(noise_level=2.62e-10)",0.649,"NN (8,)",0.057
6,25,2.0,"1.59**2 * Matern(length_scale=[0.786, 1.23, 1.3, 1.13, 1.8], nu=2.5) + WhiteKernel(noise_level=0.0382)",0.600,None,—
7,35,2.0,"0.738**2 * Matern(length_scale=[1.03, 2, 2, 0.316, 0.174, 0.121], nu=2.5) + WhiteKernel(noise_level=0.0784)",0.800,Linear regression,0.008
8,45,10.0,"3.08**2 * Matern(length_scale=[3.01, 4.68, 2.4, 5.47, 10, 4.89, 3.08, 10], nu=2.5) + WhiteKernel(noise_level=1e-10)",1.660,None,—


### Refitting the reference models and recording computational cost

This cell refits the previous round's GP configurations using all
observations available after round 5. Kernel hyperparameters are
re-estimated within the retained bounds.

The supplementary one-layer neural network for Function 5 and linear
regression for Function 7 are also refitted. Their role remains
diagnostic; no new query is selected by this cell.

Elapsed fitting time is recorded separately for the GP and auxiliary
model. These measurements describe this execution on the current
machine. They are not controlled hardware benchmarks, do not measure
energy consumption, and exclude candidate generation and prediction.

Retaining the model configurations provides continuity while allowing
the new observations to change the fitted response surfaces. It does
not imply that these configurations are optimal or that previous
uncertainty estimates were reliable.

> *Module 17 connection: Andrea Dunbar's discussion of edge AI
> highlights the importance of computational constraints alongside
> predictive performance. Recording fitting time is a first practical
> step towards considering efficiency in this BBO workflow, although
> it does not reproduce an edge deployment evaluation.*

### Interpretation of the updated model fits

All eight GP models were fitted using the observations available
after round 5. Each fit took less than two seconds in this execution.

Function 5's fitted length scales are all within the specified bounds.
Functions 1, 2, 3, 7 and 8 still have length scales at or near a bound.
These values describe the fitted models; they do not establish that
particular variables are irrelevant.

Function 7's white-noise component increased to approximately 0.0784.
This indicates that the fitted model attributes more variation to
that component. It does not prove that the underlying function is
stochastic: model mismatch can also contribute to this estimate.
Because outputs are normalised during fitting, this parameter is
expressed on the normalised output scale.

The auxiliary neural network and linear regression fitted quickly,
but timing alone does not establish their predictive usefulness.

> *Efficiency note: These measurements cover model fitting only.
> Candidate generation and prediction will be timed separately before
> drawing conclusions about the computational cost of the workflow.*

In [6]:
from scipy.spatial import cKDTree

beta_round_6 = 2.0

round_6_candidate_proposals = {}
candidate_rows = []
search_timing_rows = []

for function in range(1, 9):
    search_start = perf_counter()

    X = updated_data[function]["X"]
    y = updated_data[function]["y"]
    gp = round_6_gp_models[function]

    dimensions = X.shape[1]
    best_input = X[int(np.argmax(y))]
    best_output = float(np.max(y))

    rng = np.random.default_rng(42 + function)

    candidate_pools = {
        "Global": rng.uniform(
            0.0, 0.999999,
            size=(100000, dimensions)
        )
    }

    # Split the previous local budget between two neighbourhood sizes.
    for label, radius in [
        ("Local 0.10", 0.10),
        ("Local 0.03", 0.03)
    ]:
        lower = np.maximum(0.0, best_input - radius)
        upper = np.minimum(0.999999, best_input + radius)

        candidate_pools[label] = rng.uniform(
            lower, upper,
            size=(10000, dimensions)
        )

    observation_tree = cKDTree(X)
    round_6_candidate_proposals[function] = {}

    for source, candidates in candidate_pools.items():
        # Predict at the rounded coordinates used by the portal.
        candidates = np.unique(
            np.round(candidates, 6),
            axis=0
        )

        distances, _ = observation_tree.query(candidates)
        keep = distances > 1e-6

        candidates = candidates[keep]
        distances = distances[keep]

        if len(candidates) == 0:
            raise ValueError(
                f"No new candidates for Function {function}: {source}"
            )

        mean_batches = []
        std_batches = []

        for start in range(0, len(candidates), 5000):
            mean_batch, std_batch = gp.predict(
                candidates[start:start + 5000],
                return_std=True
            )
            mean_batches.append(mean_batch)
            std_batches.append(std_batch)

        mean = np.concatenate(mean_batches)
        std = np.concatenate(std_batches)
        ucb = mean + beta_round_6 * std

        index = int(np.argmax(ucb))
        query = candidates[index].copy()

        auxiliary_prediction = np.nan
        if function in round_6_auxiliary_models:
            auxiliary_prediction = float(
                round_6_auxiliary_models[function]
                .predict(query.reshape(1, -1))[0]
            )

        proposal = {
            "query": query,
            "mean": float(mean[index]),
            "std": float(std[index]),
            "ucb": float(ucb[index]),
            "nearest_distance": float(distances[index]),
            "auxiliary_prediction": auxiliary_prediction
        }

        round_6_candidate_proposals[function][source] = proposal

        candidate_rows.append({
            "Function": function,
            "Search": source,
            "Query": "-".join(f"{value:.6f}" for value in query),
            "Current best": best_output,
            "GP mean": proposal["mean"],
            "GP std": proposal["std"],
            "UCB": proposal["ucb"],
            "Nearest observation distance": proposal["nearest_distance"],
            "Auxiliary prediction": auxiliary_prediction
        })

    search_timing_rows.append({
        "Function": function,
        "Candidate search seconds": perf_counter() - search_start
    })

round_6_candidate_comparison = (
    pd.DataFrame(candidate_rows)
    .set_index(["Function", "Search"])
)

round_6_search_times = (
    pd.DataFrame(search_timing_rows)
    .set_index("Function")
)

display(
    round_6_candidate_comparison.style.format({
        "Current best": "{:.6e}",
        "GP mean": "{:.6e}",
        "GP std": "{:.6e}",
        "UCB": "{:.6e}",
        "Nearest observation distance": "{:.6f}",
        "Auxiliary prediction": "{:.6e}"
    }, na_rep="—")
)

display(
    round_6_search_times.style.format({
        "Candidate search seconds": "{:.3f}"
    })
)

,Candidate search seconds
Function,
1,0.496
2,0.409
3,0.514
4,0.746
5,0.708
6,0.607
7,0.723
8,0.921


### Comparing global search with two local scales

This cell evaluates 100,000 global candidates and two sets of 10,000
local candidates per function. The local boxes extend up to 0.10 and
0.03 per coordinate around the best observed input, restricted to
the permitted domain.

The total candidate budget remains 120,000 before rounding and
duplicate removal, matching the previous round. The change allocates
half of the local budget to a more concentrated neighbourhood.

All three searches use the same fitted GP and UCB rule. The
highest-scoring candidate from each pool is retained for comparison.
Auxiliary predictions are recorded for Functions 5 and 7 but do not
alter the acquisition scores.

This approach combines broad coverage with more concentrated sampling
near demonstrated performance. The smaller local box can investigate
fine adjustments, but neither local search guarantees an improvement.

Search time includes candidate generation, rounding, distance checks,
predictions and proposal selection. It excludes model fitting and
displaying the tables.

> *Module 17 connection: Comparing broad and fine search scales offers
> a procedural analogy to progressively examining patterns at different
> levels of detail. It is not CNN feature extraction: no convolution,
> pooling or learned feature hierarchy is implemented here.*

> *Methodological note: The neighbourhood sizes are heuristics. The
> comparison does not isolate their causal effect, because the fitted
> models and available observations have also changed since round 5.
> No final query is selected or submitted by this cell.*

In [7]:
# Select the highest-UCB proposal across all three search scales.
# Preserve predictions before receiving round 6 results.

round_6_selected = {}
round_6_queries = {}
submission_rows = []

for function in range(1, 9):
    proposals = round_6_candidate_proposals[function]

    selected_search = max(
        proposals,
        key=lambda source: proposals[source]["ucb"]
    )

    proposal = proposals[selected_search]
    query = proposal["query"].copy()
    X = updated_data[function]["X"]

    # Validate the exact coordinates to be submitted.
    assert query.shape == (X.shape[1],)
    assert np.isfinite(query).all()
    assert ((query >= 0.0) & (query < 1.0)).all()
    assert np.min(np.linalg.norm(X - query, axis=1)) > 1e-6

    portal_query = "-".join(f"{value:.6f}" for value in query)

    round_6_selected[function] = {
        **proposal,
        "query": query,
        "search": selected_search,
        "model": round_6_gp_models[function],
        "length_scale_upper_bound": gp_upper_bounds[function],
        "auxiliary_model": round_6_auxiliary_models.get(function),
        "portal_query": portal_query
    }

    round_6_queries[function] = query.copy()

    submission_rows.append({
        "Function": function,
        "Selected search": selected_search,
        "Query": portal_query,
        "Predicted mean": proposal["mean"],
        "Predicted std": proposal["std"],
        "UCB": proposal["ucb"],
        "Auxiliary prediction": proposal["auxiliary_prediction"]
    })

round_6_submission = (
    pd.DataFrame(submission_rows)
    .set_index("Function")
)

display(
    round_6_submission.style.format({
        "Predicted mean": "{:.6e}",
        "Predicted std": "{:.6e}",
        "UCB": "{:.6e}",
        "Auxiliary prediction": "{:.6e}"
    }, na_rep="—")
)

# Compact table for copying into the portal.
final_queries_table = round_6_submission[["Query"]].rename(
    columns={"Query": "Query for submission"}
)

with pd.option_context("display.max_colwidth", None):
    display(final_queries_table)

,Selected search,Query,Predicted mean,Predicted std,UCB,Auxiliary prediction
Function,,,,,,
1,Global,0.123843-0.121237,-2.106466e-04,9.522384e-04,1.693830e-03,—
2,Global,0.709248-0.306664,6.570358e-01,3.125545e-02,7.195467e-01,—
3,Global,0.045627-0.009123-0.525961,-5.480886e-02,6.313081e-02,7.145277e-02,—
4,Local 0.03,0.424420-0.403134-0.390867-0.438795,5.735155e-02,3.154412e-01,6.882339e-01,—
5,Global,0.994128-0.987118-0.927371-0.967531,4.550247e+03,5.910056e+02,5.732259e+03,4.910054e+03
6,Global,0.432285-0.374372-0.559617-0.651650-0.005539,-5.016286e-01,1.660142e-01,-1.696001e-01,—
7,Local 0.03,0.000293-0.248999-0.558162-0.154628-0.355919-0.762975,1.781922e+00,1.693531e-01,2.120628e+00,7.990466e-01
8,Global,0.086109-0.035213-0.037529-0.621722-0.929968-0.637726-0.311255-0.202704,9.813935e+00,1.641871e-01,1.014231e+01,—


,Query for submission
Function,
1,0.123843-0.121237
2,0.709248-0.306664
3,0.045627-0.009123-0.525961
4,0.424420-0.403134-0.390867-0.438795
5,0.994128-0.987118-0.927371-0.967531
6,0.432285-0.374372-0.559617-0.651650-0.005539
7,0.000293-0.248999-0.558162-0.154628-0.355919-0.762975
8,0.086109-0.035213-0.037529-0.621722-0.929968-0.637726-0.311255-0.202704


### Final query selection — round 6

For each function, I selected the proposal with the highest GP-UCB
score across the global search and the two local search scales.

The selected proposals come from the global pool for Functions
1, 2, 3, 5, 6 and 8, and from the local 0.03 pool for Functions 4
and 7. The additional fine-scale search therefore contributed two
selected queries. This does not yet demonstrate that it improves
actual optimisation performance.

For Function 5, the global proposal has a lower GP mean than the
local 0.10 proposal but substantially greater uncertainty, resulting
in a higher UCB. Its auxiliary neural-network prediction exceeds
the current best, whereas its GP mean does not. This selection
remains an exploratory experiment.

Function 8 also favours a global proposal through its uncertainty
bonus, despite higher predicted means for the local alternatives.
This accepts the possibility of a short-term loss to investigate
a less familiar region.

For Function 7, the selected local point has predictions of approximately
1.78 from the GP and 0.80 from linear regression. The disagreement
is recorded for subsequent evaluation; neither model establishes
the true outcome.

Candidate search took less than one second per function in this
execution. Together with the recorded fitting times, this suggests
that the current workflow is computationally manageable on the
present machine. No energy or deployment-latency measurements
were performed.

The selected coordinates and historical predictions are retained
in notebook memory and displayed in the output tables. This cell
does not submit queries to the portal.

> *Methodological note: The acquisition rule remains mean plus twice
> the predictive standard deviation. Auxiliary predictions do not
> change the selection, and UCB is not a guaranteed upper bound on
> the unknown function value. Actual sixth-round outcomes are needed
> to evaluate these decisions.*